# 46 â€” Siamese Cliff Net

Siamese neural network trained on cliff pairs via a combined regression + contrastive loss.
The encoder learns embeddings where cliff-active and cliff-inactive compounds are well-separated.

**Primary metric:** RAE (lower is better). Current best OOF RAE: 0.5281.

In [1]:
import os as _os
_torch_lib = r"d:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\.venv\Lib\site-packages\torch\lib"
if _os.path.exists(_torch_lib):
    _os.add_dll_directory(_torch_lib)

import sys, os
os.environ["PYTHONIOENCODING"] = "utf-8"
sys.path.insert(0, "../src")
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, Dataset
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV
import lightgbm as lgb
from pxr.data import load_train, load_test
from pxr.featurize import combined, impute
from pxr.eval import rae, scaffold_kfold_indices
from pxr.chem import bemis_murcko, morgan_fp_batch
from pxr.paths import DATA_PROCESSED, SUBMISSIONS

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
LGBM_PARAMS = dict(n_estimators=1000, num_leaves=64, learning_rate=0.05,
                   subsample=0.8, colsample_bytree=0.8, reg_alpha=0.1,
                   reg_lambda=0.1, min_child_samples=10, n_jobs=4, verbose=-1)
print(f"Device: {DEVICE}")

Device: cpu


## 1. Load data

In [2]:
tr = load_train()
te = load_test()
print(f"Train: {len(tr)} | Test: {len(te)}")

tr['scaffold'] = tr['smiles'].apply(bemis_murcko)

# Raw features (will be standardized per fold)
X_tr_raw = combined(tr['smiles'].tolist())
X_te_raw = combined(te['smiles'].tolist())
X_tr_raw = impute(X_tr_raw)
X_te_raw = impute(X_te_raw)
y_tr = tr['pec50'].values.astype(np.float32)
print(f"X_tr: {X_tr_raw.shape} | X_te: {X_te_raw.shape}")

Train: 4139 | Test: 513


X_tr: (4139, 2265) | X_te: (513, 2265)


## 2. Build cliff pair dataset

In [3]:
def compute_cliff_pairs(smiles_list, pec50_arr, sim_thresh=0.5, delta_thresh=1.0):
    """Return list of (idx_active, idx_inactive, delta) pairs."""
    fps = morgan_fp_batch(smiles_list).astype(np.float32)
    dot = fps @ fps.T
    rowsum = fps.sum(1)
    union = rowsum[:, None] + rowsum[None, :] - dot
    tanimoto_mat = dot / union.clip(min=1)
    np.fill_diagonal(tanimoto_mat, 0)

    pairs = []
    rows, cols = np.where((tanimoto_mat >= sim_thresh))
    for i, j in zip(rows, cols):
        if i >= j:
            continue
        delta = pec50_arr[i] - pec50_arr[j]
        if abs(delta) >= delta_thresh:
            if delta > 0:
                pairs.append((i, j, delta))
            else:
                pairs.append((j, i, -delta))
    return pairs

all_pairs = compute_cliff_pairs(tr['smiles'].tolist(), y_tr, sim_thresh=0.5, delta_thresh=1.0)
print(f"Total cliff pairs (sim>=0.5, |Î”|>=1.0): {len(all_pairs)}")
if len(all_pairs) > 0:
    print(f"  delta range: {min(p[2] for p in all_pairs):.2f} â€” {max(p[2] for p in all_pairs):.2f}")

Total cliff pairs (sim>=0.5, |Î”|>=1.0): 148
  delta range: 1.00 â€” 3.32


## 3. Siamese architecture

In [4]:
class SiameseEncoder(nn.Module):
    def __init__(self, d_in=2265, d_emb=256):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(d_in, 512), nn.LayerNorm(512), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(512, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(0.2),
            nn.Linear(256, d_emb)
        )
        self.regressor = nn.Linear(d_emb, 1)

    def encode(self, x):
        return self.encoder(x)

    def forward(self, x):
        return self.regressor(self.encode(x)).squeeze(-1)


class PairDataset(Dataset):
    """Dataset that yields (x_active, x_inactive, margin) tuples for contrastive loss."""
    def __init__(self, X, pairs):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.pairs = pairs  # list of (ia, ii, delta)

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        ia, ii, delta = self.pairs[idx]
        return self.X[ia], self.X[ii], torch.tensor(delta, dtype=torch.float32)


print("SiameseEncoder defined.")
d_in = X_tr_raw.shape[1]
test_model = SiameseEncoder(d_in=d_in, d_emb=256)
n_params = sum(p.numel() for p in test_model.parameters())
print(f"Model parameters: {n_params:,}")

SiameseEncoder defined.
Model parameters: 1,359,105


## 4. Training with combined regression + contrastive loss

In [5]:
def train_siamese(
    X_train, y_train, pairs,
    d_emb=256,
    n_epochs=100,
    batch_size=128,
    lr=1e-3,
    lambda_contrastive=0.5,
    contrastive_margin=0.5,
    device=DEVICE,
):
    d_in = X_train.shape[1]
    scaler = StandardScaler()
    X_sc = scaler.fit_transform(X_train).astype(np.float32)

    model = SiameseEncoder(d_in=d_in, d_emb=d_emb).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
    huber = nn.HuberLoss(delta=1.0)

    # Individual compound dataset
    X_t = torch.tensor(X_sc, dtype=torch.float32).to(device)
    y_t = torch.tensor(y_train, dtype=torch.float32).to(device)
    indiv_ds = TensorDataset(X_t, y_t)
    indiv_loader = DataLoader(indiv_ds, batch_size=batch_size, shuffle=True)

    # Pair dataset for contrastive loss
    pair_loader = None
    if len(pairs) > 0:
        pair_ds = PairDataset(X_sc, pairs)
        pair_loader = DataLoader(pair_ds, batch_size=min(batch_size, len(pairs)),
                                 shuffle=True, drop_last=False)

    model.train()
    for epoch in range(n_epochs):
        total_loss = 0.0
        n_batches = 0

        # Regression pass on individual compounds
        for x_batch, y_batch in indiv_loader:
            optimizer.zero_grad()
            pred = model(x_batch)
            reg_loss = huber(pred, y_batch)

            # Contrastive loss on pairs (if available)
            contr_loss = torch.tensor(0.0, device=device)
            if pair_loader is not None:
                try:
                    xa, xi, margins = next(iter(pair_loader))
                    xa, xi = xa.to(device), xi.to(device)
                    pred_a = model(xa)
                    pred_i = model(xi)
                    # active should be > inactive by at least contrastive_margin
                    contr_loss = torch.clamp(
                        contrastive_margin - (pred_a - pred_i), min=0
                    ).pow(2).mean()
                except StopIteration:
                    pass

            loss = reg_loss + lambda_contrastive * contr_loss
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            n_batches += 1

        scheduler.step()
        if (epoch + 1) % 25 == 0:
            print(f"    Epoch {epoch+1}/{n_epochs}  avg_loss={total_loss/max(n_batches,1):.4f}")

    return model, scaler


@torch.no_grad()
def predict_siamese(model, scaler, X, batch_size=256, device=DEVICE):
    model.eval()
    X_sc = scaler.transform(X).astype(np.float32)
    X_t = torch.tensor(X_sc, dtype=torch.float32)
    preds = []
    for i in range(0, len(X_t), batch_size):
        batch = X_t[i:i+batch_size].to(device)
        preds.append(model(batch).cpu().numpy())
    return np.concatenate(preds)

print("Training utilities defined.")

Training utilities defined.


## 5. Scaffold 5-fold CV

In [6]:
splits = scaffold_kfold_indices(tr['scaffold'], n_splits=5)
oof_preds = np.full(len(tr), np.nan)
fold_raes = []

for fold, (tr_idx, val_idx) in enumerate(splits):
    print(f"\nFold {fold+1}/5 â€” train={len(tr_idx)}, val={len(val_idx)}")

    X_fold_tr = X_tr_raw[tr_idx]
    X_fold_val = X_tr_raw[val_idx]
    y_fold_tr = y_tr[tr_idx]
    y_fold_val = y_tr[val_idx]

    # Cliff pairs within this fold's training data
    fold_pairs = [(ia, ii, d) for ia, ii, d in all_pairs
                  if ia in set(tr_idx) and ii in set(tr_idx)]
    # Re-index to local
    idx_map = {orig: local for local, orig in enumerate(tr_idx)}
    fold_pairs_local = [(idx_map[ia], idx_map[ii], d) for ia, ii, d in fold_pairs]
    print(f"  Cliff pairs in fold: {len(fold_pairs_local)}")

    model, scaler = train_siamese(
        X_fold_tr, y_fold_tr, fold_pairs_local,
        n_epochs=100, batch_size=128, lr=1e-3,
        lambda_contrastive=0.5, contrastive_margin=0.5,
    )

    val_preds = predict_siamese(model, scaler, X_fold_val)
    fold_rae = rae(y_fold_val, val_preds)
    fold_raes.append(fold_rae)
    oof_preds[val_idx] = val_preds
    print(f"  Fold RAE: {fold_rae:.4f}")

oof_rae = rae(y_tr, oof_preds)
print(f"\n=== OOF RAE: {oof_rae:.4f} (mean fold: {np.mean(fold_raes):.4f} Â± {np.std(fold_raes):.4f}) ===")


Fold 1/5 â€” train=3311, val=828
  Cliff pairs in fold: 88


    Epoch 25/100  avg_loss=0.0249


    Epoch 50/100  avg_loss=0.0140


    Epoch 75/100  avg_loss=0.0107


    Epoch 100/100  avg_loss=0.0094
  Fold RAE: 0.5264

Fold 2/5 â€” train=3311, val=828
  Cliff pairs in fold: 98


...

    Epoch 100/100  avg_loss=0.0087
  Fold RAE: 0.6087

Fold 5/5 â€” train=3312, val=827
  Cliff pairs in fold: 100


    Epoch 25/100  avg_loss=0.0221


    Epoch 50/100  avg_loss=0.0149


    Epoch 75/100  avg_loss=0.0107


    Epoch 100/100  avg_loss=0.0100
  Fold RAE: 0.6455

=== OOF RAE: 0.5943 (mean fold: 0.5995 Â± 0.0394) ===


## 6. Final model â€” train on all data, predict test

In [7]:
print("Training final model on all training data...")
# Convert global pairs to local indices (they already are global indices matching X_tr_raw)
final_model, final_scaler = train_siamese(
    X_tr_raw, y_tr, all_pairs,
    n_epochs=100, batch_size=128, lr=1e-3,
    lambda_contrastive=0.5, contrastive_margin=0.5,
)

test_preds = predict_siamese(final_model, final_scaler, X_te_raw)

# Clip to training range Â± 0.5
y_lo = y_tr.min() - 0.5
y_hi = y_tr.max() + 0.5
test_preds = np.clip(test_preds, y_lo, y_hi)

print(f"Test pred range: [{test_preds.min():.3f}, {test_preds.max():.3f}]")

Training final model on all training data...


    Epoch 25/100  avg_loss=0.0220


    Epoch 50/100  avg_loss=0.0143


    Epoch 75/100  avg_loss=0.0102


    Epoch 100/100  avg_loss=0.0086
Test pred range: [2.199, 6.135]


In [8]:
# Save OOF
np.save(DATA_PROCESSED / 'oof_siamese_cliff.npy', oof_preds)
print("Saved OOF predictions.")

# Save submission
sub = pd.DataFrame({'Molecule Name': te['name'], 'pEC50': test_preds})
out_path = SUBMISSIONS / '46_siamese_cliff_net.csv'
sub.to_csv(out_path, index=False)
print(f"Saved submission to {out_path}")
print(sub.head())
print(f"\nFinal OOF RAE: {oof_rae:.4f}")

Saved OOF predictions.
Saved submission to D:\Users\ashenoy00000\.windsurf\OpenADMET-pxr-challenge\submissions\46_siamese_cliff_net.csv
    Molecule Name     pEC50
0  OADMET-0006617  4.398894
1  OADMET-0006616  4.453000
2  OADMET-0006615  5.550648
3  OADMET-0006614  5.787705
4  OADMET-0006613  4.971629

Final OOF RAE: 0.5943
